# Task 4: General Health Query Chatbot
**DevelopersHub Corporation — AI/ML Engineering Internship**

## Problem Statement
Build a conversational chatbot that answers general health-related questions using a Large Language Model (LLM). The chatbot must respond in a friendly, clear manner while avoiding harmful medical advice.

## Approach
- **LLM:** LLaMA 3.3 70B via Groq API (fast inference, free tier)
- **Technique:** Prompt Engineering — system prompt design to control tone, safety, and response quality
- **Safety:** Built-in filters to block dangerous medical advice
- **Interface:** Interactive chat loop in notebook + clean conversation history

## Key Skills Demonstrated
- Prompt design and testing
- LLM API integration (Groq)
- Safety handling in chatbot responses
- Conversational agent design with memory

## Step 1: Install & Import Libraries

In [1]:
# Install required libraries
%pip install groq -q

import os
from groq import Groq
from IPython.display import display, Markdown
import textwrap

print('All libraries imported successfully!')


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
All libraries imported successfully!


## Step 2: Configure Groq API

In [ ]:
import os
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# Model selection
MODEL = "llama-3.3-70b-versatile"

print(f'Groq client initialized!')
print(f'Model: {MODEL}')

Groq client initialized!
Model: llama-3.3-70b-versatile


## Step 3: Prompt Engineering — System Prompt Design

The system prompt is the most critical part of a health chatbot. It defines:
1. **Persona** — who the bot is
2. **Tone** — friendly, clear, empathetic
3. **Safety rules** — what it must NEVER do
4. **Scope** — general health education only, not diagnosis

In [3]:
# === SYSTEM PROMPT — Core of prompt engineering ===
SYSTEM_PROMPT = """
You are HealthBot, a friendly and knowledgeable general health assistant.
Your role is to provide clear, accurate, and easy-to-understand health information
to help people better understand their health and wellness.

## Your Personality:
- Warm, empathetic, and supportive
- Use simple language — avoid heavy medical jargon
- Be concise but thorough
- Always end responses with encouragement to consult a doctor when relevant

## What You CAN Do:
- Explain symptoms, conditions, and diseases in simple terms
- Provide general wellness and prevention tips
- Explain how common medications work (general info only)
- Suggest when someone should seek medical attention
- Answer questions about nutrition, exercise, and mental wellness

## STRICT Safety Rules — You MUST follow these:
1. NEVER diagnose a specific medical condition for the user
2. NEVER prescribe medications or specific dosages
3. NEVER tell a user NOT to see a doctor
4. NEVER provide advice for emergencies — always direct to emergency services (115 in Pakistan)
5. If a question is beyond general health education, say: "This is a question best answered by a qualified doctor."
6. NEVER provide advice that could replace professional medical consultation

## Response Format:
- Start with a direct, helpful answer
- Use bullet points for lists
- End with: "💡 Remember: Always consult a healthcare professional for personal medical advice."
- Keep responses under 300 words unless the topic requires more detail
"""

print('System prompt configured!')
print(f'Prompt length: {len(SYSTEM_PROMPT)} characters')

System prompt configured!
Prompt length: 1480 characters


## Step 4: Safety Filter — Blocking Harmful Queries

In [4]:
# === Safety Filter ===
# These keywords trigger an automatic safe response without calling the LLM

BLOCKED_KEYWORDS = [
    "suicide", "self harm", "self-harm", "kill myself", "overdose",
    "how to die", "end my life", "poison myself", "hurt myself"
]

EMERGENCY_KEYWORDS = [
    "chest pain", "heart attack", "can't breathe", "cannot breathe",
    "stroke", "unconscious", "not breathing", "severe bleeding"
]

SAFE_RESPONSE = """
I'm really sorry you're going through this. Your life has value and there is help available.

Please reach out immediately:
- **Pakistan Mental Health Helpline:** 0311-7786264
- **Umang Helpline:** 0317-4288665
- **Emergency Services:** 115

You don't have to face this alone. Please talk to someone you trust or a mental health professional.
"""

EMERGENCY_RESPONSE = """
⚠️ This sounds like a medical emergency!

**Please call emergency services IMMEDIATELY:**
- **Pakistan Emergency:** 115 (Rescue)
- **Edhi Foundation:** 115

Do not wait — get to the nearest hospital or call for help right now.
"""

def safety_check(user_input):
    """
    Check if user input contains blocked or emergency keywords.
    Returns (is_safe, response_if_unsafe)
    """
    input_lower = user_input.lower()
    
    # Check for self-harm / crisis keywords
    for keyword in BLOCKED_KEYWORDS:
        if keyword in input_lower:
            return False, SAFE_RESPONSE
    
    # Check for emergency keywords
    for keyword in EMERGENCY_KEYWORDS:
        if keyword in input_lower:
            return False, EMERGENCY_RESPONSE
    
    return True, None

print('Safety filter configured!')
print(f'Blocked keywords: {len(BLOCKED_KEYWORDS)}')
print(f'Emergency keywords: {len(EMERGENCY_KEYWORDS)}')

Safety filter configured!
Blocked keywords: 9
Emergency keywords: 8


## Step 5: Core Chatbot Function

In [5]:
# === Core chat function with conversation memory ===

def chat(user_message, conversation_history):
    """
    Send a message to HealthBot and get a response.
    Maintains conversation history for context-aware replies.
    
    Args:
        user_message (str): The user's health question
        conversation_history (list): Previous messages in the conversation
    
    Returns:
        tuple: (assistant_response, updated_history)
    """
    
    # Step 1: Run safety check BEFORE calling LLM
    is_safe, unsafe_response = safety_check(user_message)
    if not is_safe:
        return unsafe_response, conversation_history
    
    # Step 2: Add user message to history
    conversation_history.append({
        "role": "user",
        "content": user_message
    })
    
    # Step 3: Call Groq API with full conversation history
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT}
        ] + conversation_history,
        max_tokens=500,
        temperature=0.7,  # Slight creativity for natural responses
    )
    
    # Step 4: Extract response
    assistant_message = response.choices[0].message.content
    
    # Step 5: Add assistant response to history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })
    
    return assistant_message, conversation_history


def display_response(user_q, bot_response):
    """Pretty print the conversation in notebook."""
    print(f"\n{'='*60}")
    print(f"👤 USER: {user_q}")
    print(f"{'='*60}")
    print(f"🤖 HEALTHBOT:\n")
    print(textwrap.fill(bot_response, width=70, replace_whitespace=False))
    print(f"{'='*60}\n")


print('Chat function ready!')

Chat function ready!


## Step 6: Test the Chatbot — Example Queries

Testing with the exact example queries mentioned in the task requirements.

In [6]:
# Initialize fresh conversation history
history = []

# === Test Query 1 (from task requirements) ===
query1 = "What causes a sore throat?"
response1, history = chat(query1, history)
display_response(query1, response1)


👤 USER: What causes a sore throat?
🤖 HEALTHBOT:

A sore throat can be caused by a variety of factors, including:
*
Viral infections such as the common cold, flu, or mononucleosis
*
Bacterial infections like strep throat
* Allergies, which can lead to
postnasal drip and throat irritation
* Dry air, which can dry out the
throat and make it sore
* Shouting, screaming, or other loud talking,
which can strain the vocal cords
* Acid reflux, where stomach acid
flows up into the throat and causes irritation
* Smoking or exposure
to secondhand smoke, which can irritate the throat

In some cases, a
sore throat can also be a symptom of a more serious underlying
condition. If you're experiencing a severe sore throat, difficulty
swallowing, or a fever over 101.5°F (38.6°C), it's a good idea to seek
medical attention.

It's also important to practice good hygiene, such
as washing your hands frequently, to help prevent the spread of
infections that can cause a sore throat.

💡 Remember: Always consul

In [7]:
# === Test Query 2 (from task requirements) ===
query2 = "Is paracetamol safe for children?"
response2, history = chat(query2, history)
display_response(query2, response2)


👤 USER: Is paracetamol safe for children?
🤖 HEALTHBOT:

Paracetamol (also known as acetaminophen) can be safe for children
when used correctly. However, it's essential to follow the recommended
dosage and guidelines. Here are some key points to consider:
* Always
read and follow the label instructions
* Use the correct dosage based
on your child's age and weight
* Never give more than the recommended
dose
* Be aware of any potential interactions with other medications
your child is taking
* If your child has any underlying medical
conditions, consult with a doctor before giving paracetamol

It's also
important to note that paracetamol should not be given to children
under 3 months old without consulting a doctor. For children over 3
months, you can give paracetamol to help relieve pain and reduce
fever, but always follow the recommended dosage.

If you're unsure
about giving paracetamol to your child or have concerns about their
symptoms, it's always best to consult with a doctor or p

In [8]:
# === Test Query 3 — General wellness ===
query3 = "What are the best foods to eat for a healthy heart?"
response3, history = chat(query3, history)
display_response(query3, response3)


👤 USER: What are the best foods to eat for a healthy heart?
🤖 HEALTHBOT:

Eating a balanced diet rich in whole foods can help support a healthy
heart. Some of the best foods for heart health include:
* Fatty fish
like salmon, tuna, and mackerel, which are high in omega-3 fatty acids
* Leafy green vegetables like spinach, kale, and collard greens, which
are rich in antioxidants and fiber
* Berries, such as blueberries,
strawberries, and raspberries, which are high in antioxidants and
fiber
* Avocados, which are a good source of healthy fats and fiber
*
Nuts and seeds, like almonds, walnuts, and chia seeds, which are rich
in healthy fats and antioxidants
* Whole grains, such as brown rice,
quinoa, and whole-wheat bread, which are high in fiber and nutrients
*
Legumes, like lentils, chickpeas, and black beans, which are rich in
protein, fiber, and minerals

It's also important to limit or avoid
foods that can increase heart disease risk, such as:
* Foods high in
saturated and trans fats,

In [9]:
# === Test Query 4 — Mental health ===
query4 = "How can I manage stress and anxiety naturally?"
response4, history = chat(query4, history)
display_response(query4, response4)


👤 USER: How can I manage stress and anxiety naturally?
🤖 HEALTHBOT:

Managing stress and anxiety naturally can be achieved through a
combination of lifestyle changes and self-care techniques. Here are
some effective ways to do so:
* Practice relaxation techniques, such
as deep breathing, meditation, or yoga
* Engage in regular exercise,
like walking, jogging, or swimming, to release endorphins and improve
mood
* Get enough sleep, aiming for 7-8 hours per night, to help
regulate stress hormones
* Connect with nature, whether it's walking
in a park or simply spending time outdoors
* Try journaling or writing
down your thoughts and feelings to process and release emotions
*
Prioritize social connections and build a support network of friends
and family
* Limit caffeine and sugar intake, as they can exacerbate
anxiety and stress
* Try herbal supplements like ashwagandha,
chamomile, or lavender, but always consult with a doctor before adding
new supplements to your routine

Remember, every

In [10]:
# === Test Query 5 — Context awareness (follow-up question) ===
# This tests if the chatbot remembers previous context
query5 = "Can you tell me more about the breathing exercises you mentioned?"
response5, history = chat(query5, history)
display_response(query5, response5)


👤 USER: Can you tell me more about the breathing exercises you mentioned?
🤖 HEALTHBOT:

Breathing exercises, also known as respiratory therapy, can be a
powerful tool to help manage stress and anxiety. Here are a few
techniques to get you started:
* Diaphragmatic breathing: Also known
as belly breathing, this involves breathing deeply into your
diaphragm, rather than shallowly into your chest. To do this, place
one hand on your belly and the other on your chest. Inhale slowly
through your nose, allowing your belly to rise as your diaphragm
descends. Your chest should not move.
* 4-7-8 breathing: This
technique involves breathing in through your nose for a count of 4,
holding your breath for a count of 7, and exhaling through your mouth
for a count of 8. This can help slow down your heart rate and promote
relaxation.
* Box breathing: This involves breathing in for a count of
4, holding your breath for a count of 4, exhaling for a count of 4,
and holding your breath again for a count of

## Step 7: Safety Filter Testing

In [11]:
# === Test Safety Filter — Emergency Query ===
# This should NOT call the LLM — safety filter intercepts it
print('Testing Safety Filter...')
print()

emergency_query = "I have severe chest pain right now"
is_safe, unsafe_resp = safety_check(emergency_query)

print(f"Query: '{emergency_query}'")
print(f"Safety Check Result: {'✅ SAFE' if is_safe else '🚨 BLOCKED — Emergency Detected'}")
if not is_safe:
    print(f"\nSafe Response Returned:\n{unsafe_resp}")

Testing Safety Filter...

Query: 'I have severe chest pain right now'
Safety Check Result: 🚨 BLOCKED — Emergency Detected

Safe Response Returned:

⚠️ This sounds like a medical emergency!

**Please call emergency services IMMEDIATELY:**
- **Pakistan Emergency:** 115 (Rescue)
- **Edhi Foundation:** 115

Do not wait — get to the nearest hospital or call for help right now.



In [12]:
# === Demonstrate conversation history ===
print('=== Conversation History Summary ===')
print(f'Total turns: {len(history) // 2} question-answer pairs')
print()
for i, msg in enumerate(history):
    role = '👤 User' if msg['role'] == 'user' else '🤖 HealthBot'
    preview = msg['content'][:80] + '...' if len(msg['content']) > 80 else msg['content']
    print(f"{i+1}. {role}: {preview}")
    print()

=== Conversation History Summary ===
Total turns: 5 question-answer pairs

1. 👤 User: What causes a sore throat?

2. 🤖 HealthBot: A sore throat can be caused by a variety of factors, including:
* Viral infectio...

3. 👤 User: Is paracetamol safe for children?

4. 🤖 HealthBot: Paracetamol (also known as acetaminophen) can be safe for children when used cor...

5. 👤 User: What are the best foods to eat for a healthy heart?

6. 🤖 HealthBot: Eating a balanced diet rich in whole foods can help support a healthy heart. Som...

7. 👤 User: How can I manage stress and anxiety naturally?

8. 🤖 HealthBot: Managing stress and anxiety naturally can be achieved through a combination of l...

9. 👤 User: Can you tell me more about the breathing exercises you mentioned?

10. 🤖 HealthBot: Breathing exercises, also known as respiratory therapy, can be a powerful tool t...



## Step 8: Interactive Chat Loop

Run this cell for a live chat session with HealthBot. Type `quit` to exit.

In [ ]:
# === Interactive Chat Loop ===
# Run this cell for a live conversation with HealthBot

print('🏥 Welcome to HealthBot!')
print('Ask me any general health question.')
print("Type 'quit' to exit.")
print('='*50)

live_history = []  # Fresh history for live session

while True:
    user_input = input('\n👤 You: ').strip()
    
    if not user_input:
        continue
    
    if user_input.lower() in ['quit', 'exit', 'bye']:
        print('\n🤖 HealthBot: Take care and stay healthy! Goodbye! 👋')
        break
    
    response, live_history = chat(user_input, live_history)
    print(f'\n🤖 HealthBot: {response}')

🏥 Welcome to HealthBot!
Ask me any general health question.
Type 'quit' to exit.


## Step 9: Prompt Engineering Analysis

This section explains the prompt engineering decisions made in this chatbot.

In [ ]:
# === Prompt Engineering Breakdown ===

analysis = """
PROMPT ENGINEERING DECISIONS
==============================

1. PERSONA DEFINITION
   - Named the bot 'HealthBot' for clear identity
   - Defined role as 'general health assistant' (not doctor)
   - This prevents the model from acting like a diagnosing physician

2. TONE CONTROL
   - Specified: warm, empathetic, simple language
   - Avoids cold clinical responses that feel impersonal
   - Instruction to avoid heavy jargon improves accessibility

3. SAFETY BOUNDARIES (Critical for medical AI)
   - Explicitly listed what bot CANNOT do (diagnose, prescribe)
   - Added emergency redirect to real services
   - Two-layer safety: keyword filter + LLM-level instruction

4. RESPONSE FORMAT
   - Bullet points for scannable answers
   - Mandatory disclaimer at end of every response
   - 300 word limit prevents information overload

5. CONVERSATION MEMORY
   - Full history passed to API on every call
   - Enables follow-up questions with context
   - More natural conversation flow

6. TEMPERATURE = 0.7
   - Slight creativity for natural responses
   - Not too high (0.9+) which could cause hallucinations
   - Not too low (0.1) which sounds robotic
"""

print(analysis)

## Step 10: Final Summary & Conclusions

### What Was Built:
A production-ready general health query chatbot powered by LLaMA 3.3 70B via Groq API, featuring:

1. **Advanced Prompt Engineering** — carefully designed system prompt controlling persona, tone, safety rules, and response format
2. **Two-Layer Safety System** — keyword-based pre-filter + LLM-level safety instructions
3. **Conversation Memory** — full history maintained for context-aware follow-up questions
4. **Emergency Handling** — automatic detection and redirection for crisis/emergency queries
5. **Interactive Interface** — both demo testing cells and live chat loop

### Key Learnings:
- Prompt engineering is critical in medical AI — a poorly designed prompt can cause real harm
- Safety must be implemented at multiple levels (pre-filter + model instruction)
- Conversation history enables more natural, context-aware interactions
- Temperature tuning affects response quality — 0.7 is optimal for health Q&A

### Limitations:
- Cannot replace professional medical advice
- LLM may occasionally hallucinate — always verify with a doctor
- Keyword filter is basic — production systems use ML-based content moderation

> **Disclaimer:** This chatbot is for educational purposes only. It does not provide medical diagnosis or treatment advice.